# ERK2 binding data: feature pipeline test

Scratch notebook to test the functions in `erk2_features.py`.
Using `data/test_molecules.csv` the SMILES are real molecules but the affinity values are made up, so don't read too much into the numbers.

In [1]:
import pandas as pd
from erk2_pipeline import clean_affinity, compute_descriptors, build_feature_table

## 1. clean_affinity

Takes messy strings, gives back (number, sign). Should return (nan, None)
for anything it can't parse.

In [2]:
test_values = ["4.2", ">10000", "<0.5", " 12 ", "1,500", "", "abc", "-3"]

for v in test_values:
    print(repr(v), "->", clean_affinity(v))

'4.2' -> (4.2, '=')
'>10000' -> (10000.0, '>')
'<0.5' -> (0.5, '<')
' 12 ' -> (12.0, '=')
'1,500' -> (1500.0, '=')
'' -> (nan, None)
'abc' -> (nan, None)
'-3' -> (nan, None)


## 2. compute_descriptors

Smiles in, properties out. Bogus SMILES should give back all NaN.

In [3]:
print("Ethanol:", compute_descriptors("CCO"))
print("Invalid:", compute_descriptors("NOT_A_SMILES"))

Ethanol: {'mol_weight': 46.069, 'logp': -0.0014000000000000123, 'tpsa': 20.23, 'h_donors': 1, 'h_acceptors': 1}
Invalid: {'mol_weight': nan, 'logp': nan, 'tpsa': nan, 'h_donors': nan, 'h_acceptors': nan}


[19:30:12] SMILES Parse Error: syntax error while parsing: NOT_A_SMILES
[19:30:12] SMILES Parse Error: check for mistakes around position 3:
[19:30:12] NOT_A_SMILES
[19:30:12] ~~^
[19:30:12] SMILES Parse Error: Failed parsing SMILES 'NOT_A_SMILES' for input: 'NOT_A_SMILES'


## 3. build_feature_table

Run the whole pipeline. Need to read affinity as a string or pandas
will eat the ">" and "<" characters before we even see them.

In [4]:
raw = pd.read_csv("data/test_molecules.csv", dtype={"affinity_nM": str})
raw

,molecule_id,smiles,affinity_nM,affinity_type
0,mol_01,CC(=O)Oc1ccccc1C(=O)O,4.2,Ki
1,mol_02,CN1C=NC2=C1C(=O)N(C(=O)N2C)C,>10000,Ki
2,mol_03,CC(C)Cc1ccc(cc1)C(C)C(=O)O,12,Kd
3,mol_04,c1ccccc1,"1,500",Ki
4,mol_05,CCO,<0.5,Kd
5,mol_06,CC(=O)Nc1ccc(O)cc1,NaN,Ki
6,mol_07,NOT_A_SMILES,25,Ki
7,mol_08,c1ccc2c(c1)cccc2,abc,Kd
8,mol_09,CCN(CC)CC,-3,Ki
9,mol_10,OC(=O)c1ccccc1O,87.5,Kd


In [5]:
features = build_feature_table(raw)
features

--- build_feature_table summary ---
Rows in: 10
Bad/missing affinity: 3
Invalid/missing SMILES: 1
Dropped (unusable): 4
Rows out: 6
  of which > or < values: 2


[19:30:12] SMILES Parse Error: syntax error while parsing: NOT_A_SMILES
[19:30:12] SMILES Parse Error: check for mistakes around position 3:
[19:30:12] NOT_A_SMILES
[19:30:12] ~~^
[19:30:12] SMILES Parse Error: Failed parsing SMILES 'NOT_A_SMILES' for input: 'NOT_A_SMILES'


,molecule_id,smiles,affinity_nM,affinity_type,affinity_sign,pKi,mol_weight,logp,tpsa,h_donors,h_acceptors
0,mol_01,CC(=O)Oc1ccccc1C(=O)O,4.2,Ki,=,8.376751,180.159,1.3101,63.60,1.0,3.0
1,mol_02,CN1C=NC2=C1C(=O)N(C(=O)N2C)C,10000.0,Ki,>,5.000000,194.194,-1.0293,61.82,0.0,3.0
2,mol_03,CC(C)Cc1ccc(cc1)C(C)C(=O)O,12.0,Kd,=,7.920819,206.285,3.0732,37.30,1.0,1.0
3,mol_04,c1ccccc1,1500.0,Ki,=,5.823909,78.114,1.6866,0.00,0.0,0.0
4,mol_05,CCO,0.5,Kd,<,9.301030,46.069,-0.0014,20.23,1.0,1.0
5,mol_10,OC(=O)c1ccccc1O,87.5,Kd,=,7.057992,138.122,1.0904,57.53,2.0,2.0


### Strict mode

Now with `exact_only=True` — drops anything that was a `>` or `<` value
instead of keeping it around.

In [6]:
strict = build_feature_table(raw, exact_only=True)
strict

--- build_feature_table summary ---
Rows in: 10
Bad/missing affinity: 3
Invalid/missing SMILES: 1
Dropped (unusable): 4
Dropped (> or < values): 2
Rows out: 4
  of which > or < values: 0


[19:30:12] SMILES Parse Error: syntax error while parsing: NOT_A_SMILES
[19:30:12] SMILES Parse Error: check for mistakes around position 3:
[19:30:12] NOT_A_SMILES
[19:30:12] ~~^
[19:30:12] SMILES Parse Error: Failed parsing SMILES 'NOT_A_SMILES' for input: 'NOT_A_SMILES'


,molecule_id,smiles,affinity_nM,affinity_type,affinity_sign,pKi,mol_weight,logp,tpsa,h_donors,h_acceptors
0,mol_01,CC(=O)Oc1ccccc1C(=O)O,4.2,Ki,=,8.376751,180.159,1.3101,63.60,1.0,3.0
1,mol_03,CC(C)Cc1ccc(cc1)C(C)C(=O)O,12.0,Kd,=,7.920819,206.285,3.0732,37.30,1.0,1.0
2,mol_04,c1ccccc1,1500.0,Ki,=,5.823909,78.114,1.6866,0.00,0.0,0.0
3,mol_10,OC(=O)c1ccccc1O,87.5,Kd,=,7.057992,138.122,1.0904,57.53,2.0,2.0
